Commands to re-name the reference data (GTEx) to match the namings of genes as of gencode v46.

Done for:
- All CIBERSORTx reference data.
- All BayesPrism reference data.
- All MuSiC reference data.

The input refers to the contents of the 20250405_TOO-Matrices file. Duplicate names are not allowed so are removed to keep the occurence with the highest counts.

In [12]:
import pandas as pd
import re
import glob
import os

# === Static setup: Load reference data once ===

# Load genes2 from simulated file
genes2 = pd.read_csv(
    "/exports/eddie/scratch/s2556897/20250616_All-Tissues-NoDup_Random_Simulated_v2_Counts.txt",
    sep='\t', usecols=[0]
)["Geneid"]
set2 = set(genes2)

# Load reference gene map
ref_path = "/exports/eddie/scratch/s2556897/20250514_Published-cfRNA/20250405_Both_LessTissueV2_2Median-Unique.tsv"
reference = pd.read_csv(ref_path, sep='\t', index_col=False)
reference.columns.values[1] = "GeneName"
reference = reference.iloc[:, :2]
reference['GeneID_clean'] = reference['GeneID'].astype(str).str.split('.').str[0]
ref_map = dict(zip(reference['GeneName'], reference['GeneID_clean']))

# Load GTF annotation and extract gene name mapping
def extract_gtf_info(attr, key):
    match = re.search(f'{key} "([^"]+)"', attr)
    return match.group(1) if match else None

gtf = pd.read_csv("/exports/eddie/scratch/s2556897/gencode.v46.annotation.gtf", sep="\t", comment="#", header=None)
gtf.columns = ['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute']
genes_gtf = gtf[gtf['feature'] == 'gene'].copy()
genes_gtf['GeneID'] = genes_gtf['attribute'].apply(lambda x: extract_gtf_info(x, "gene_id"))
genes_gtf['GeneID_clean'] = genes_gtf['GeneID'].str.split('.').str[0]
genes_gtf['GeneName'] = genes_gtf['attribute'].apply(lambda x: extract_gtf_info(x, "gene_name"))
gtf_map = dict(zip(genes_gtf['GeneID_clean'], genes_gtf['GeneName']))

# === Process each CIBERSORTx matrix file ===

input_files = glob.glob("CIBERSORTx-TOO-Matrix_*/CIBERSORTx_*.bm.K999.txt")

print(f"Found {len(input_files)} files to process.")

for filepath in input_files:
    print(f"Processing: {filepath}")

    # Step 1: Load genes from file
    genes1 = pd.read_csv(filepath, sep='\t', usecols=[0])["NAME"]
    set1 = set(genes1)
    unique_to_file1 = set1 - set2

    # Step 4 & 5: Map via reference → GTF
    unmatched_to_geneid = {g: ref_map[g] for g in unique_to_file1 if g in ref_map}
    unmatched_to_gtf_name = {
        g: gtf_map[unmatched_to_geneid[g]] 
        for g in unmatched_to_geneid 
        if unmatched_to_geneid[g] in gtf_map
    }

    print(f"  Unmatched: {len(unique_to_file1)} → Remapped: {len(unmatched_to_gtf_name)}")

    # Step 6: Load full matrix and replace gene names
    cibersortx = pd.read_csv(filepath, sep='\t')
    cibersortx['NAME'] = cibersortx['NAME'].apply(lambda x: unmatched_to_gtf_name.get(x, x))

    # Step 7: Check for duplicate gene names and keep row with highest total count
    if cibersortx['NAME'].duplicated().any():
        print("  Duplicate gene names found — deduplicating by row sum...")
        
        # Sum counts across all sample columns (excluding 'NAME')
        cibersortx['row_sum'] = cibersortx.drop(columns='NAME').sum(axis=1)

        # Keep the row with the highest sum for each duplicated NAME
        cibersortx = cibersortx.sort_values('row_sum', ascending=False).drop_duplicates(subset='NAME', keep='first')

        # Drop the helper column
        cibersortx = cibersortx.drop(columns='row_sum')

    # Step 8: Save modified file
    out_path = filepath.replace(".bm.K999.txt", ".withGTFNames.txt")
    cibersortx.to_csv(out_path, sep='\t', index=False)
    print(f"  Saved to: {out_path}")


/tmp/ipykernel_820584/4151766912.py:17: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  reference = pd.read_csv(ref_path, sep='\t', index_col=False)


Found 12 files to process.
Processing: CIBERSORTx-TOO-Matrix_2Median_1500_2000/CIBERSORTx_20250405_PhenotypeClass_LessTissueV2_2Median.CIBERSORTx_20250405_GeneID_LessTissueV2_2Median.bm.K999.txt
  Unmatched: 927 → Remapped: 890
  Duplicate gene names found — deduplicating by row sum...
  Saved to: CIBERSORTx-TOO-Matrix_2Median_1500_2000/CIBERSORTx_20250405_PhenotypeClass_LessTissueV2_2Median.CIBERSORTx_20250405_GeneID_LessTissueV2_2Median.withGTFNames.txt
Processing: CIBERSORTx-TOO-Matrix_Sampling5_300_500/CIBERSORTx_20250405_PhenotypeClass_LessTissueV2_Sampling5.CIBERSORTx_20250405_GeneID_LessTissueV2_Sampling5.bm.K999.txt
  Unmatched: 524 → Remapped: 498
  Saved to: CIBERSORTx-TOO-Matrix_Sampling5_300_500/CIBERSORTx_20250405_PhenotypeClass_LessTissueV2_Sampling5.CIBERSORTx_20250405_GeneID_LessTissueV2_Sampling5.withGTFNames.txt
Processing: CIBERSORTx-TOO-Matrix_2Median_1000_1500/CIBERSORTx_20250405_PhenotypeClass_LessTissueV2_2Median.CIBERSORTx_20250405_GeneID_LessTissueV2_2Median.bm

In [13]:
import pandas as pd
import re
import glob

# === Set paths ===
# File with all genes used in simulated data (used as reference for comparison)
simulated_counts_path = "/exports/eddie/scratch/s2556897/20250616_All-Tissues-NoDup_Random_Simulated_v2_Counts.txt"

# Reference file mapping GeneName to GeneID
ref_path = "/exports/eddie/scratch/s2556897/20250514_Published-cfRNA/20250405_Both_LessTissueV2_2Median-Unique.tsv"

# GTF file with gene annotations
gtf_path = "/exports/eddie/scratch/s2556897/gencode.v46.annotation.gtf"

# Directory with CIBERSORTx gene-renamed files
input_files = glob.glob("CIBERSORTx-TOO-Matrix_*/*.withGTFNames.txt")

# === Load simulated gene list (used as the "gold standard") ===
genes2 = pd.read_csv(simulated_counts_path, sep='\t', usecols=[0])["Geneid"]
set2 = set(genes2)

# === Load reference file with GeneName → GeneID_clean mapping ===
reference = pd.read_csv(ref_path, sep='\t', index_col=False)
reference.columns.values[1] = "GeneName"
reference = reference.iloc[:, :2]  # Keep only GeneID and GeneName
reference['GeneID_clean'] = reference['GeneID'].astype(str).str.split('.').str[0]
gene_name_to_id = dict(zip(reference['GeneName'], reference['GeneID_clean']))

# === Load and process the GTF file ===
gtf = pd.read_csv(gtf_path, sep="\t", comment="#", header=None, low_memory=False)
gtf.columns = ['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute']

# Function to extract version-less gene_id from attribute field
def extract_gene_id_clean(attr):
    match = re.search('gene_id "([^"]+)"', attr)
    if match:
        return match.group(1).split('.')[0]
    return None

# Add GeneID_clean column
gtf['GeneID_clean'] = gtf['attribute'].apply(extract_gene_id_clean)
genes_gtf = gtf[gtf['feature'] == 'gene']

# === Iterate over each renamed CIBERSORTx matrix and analyze gene overlaps ===
for file in input_files:
    print(f"\nProcessing file: {file}")
    
    # Load gene list from current file 
    try:
        genes1 = pd.read_csv(file, sep='\t', usecols=[0])["NAME"]
    except Exception as e:
        print(f"  Skipped due to read error: {e}")
        continue

    set1 = set(genes1)

    # Identify shared and unique genes
    shared_genes = set1.intersection(set2)
    unique_to_file1 = set1 - set2

    print(f"  Total genes in file: {len(set1)}")
    print(f"  Shared with simulated: {len(shared_genes)}")
    print(f"  Not in simulated: {len(unique_to_file1)}")

    # Try to map unique genes to GeneID_clean using reference
    unmatched_genes_ids = [gene_name_to_id[gn] for gn in unique_to_file1 if gn in gene_name_to_id]

    # Filter GTF for those GeneID_clean values
    gtf_matched = genes_gtf[genes_gtf['GeneID_clean'].isin(unmatched_genes_ids)]
    print(f"  Unique genes mapped to GTF GeneIDs: {len(gtf_matched)}")


/tmp/ipykernel_820584/62513747.py:23: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  reference = pd.read_csv(ref_path, sep='\t', index_col=False)



Processing file: CIBERSORTx-TOO-Matrix_2Median_1500_2000/CIBERSORTx_20250405_PhenotypeClass_LessTissueV2_2Median.CIBERSORTx_20250405_GeneID_LessTissueV2_2Median.withGTFNames.txt
  Total genes in file: 8930
  Shared with simulated: 8893
  Not in simulated: 37
  Unique genes mapped to GTF GeneIDs: 0

Processing file: CIBERSORTx-TOO-Matrix_Sampling5_300_500/CIBERSORTx_20250405_PhenotypeClass_LessTissueV2_Sampling5.CIBERSORTx_20250405_GeneID_LessTissueV2_Sampling5.withGTFNames.txt
  Total genes in file: 6501
  Shared with simulated: 6475
  Not in simulated: 26
  Unique genes mapped to GTF GeneIDs: 0

Processing file: CIBERSORTx-TOO-Matrix_2Median_1000_1500/CIBERSORTx_20250405_PhenotypeClass_LessTissueV2_2Median.CIBERSORTx_20250405_GeneID_LessTissueV2_2Median.withGTFNames.txt
  Total genes in file: 8590
  Shared with simulated: 8553
  Not in simulated: 37
  Unique genes mapped to GTF GeneIDs: 0

Processing file: CIBERSORTx-TOO-Matrix_Sampling10_500_1000/CIBERSORTx_20250405_PhenotypeClass_L

In [15]:
import pandas as pd
import re
import glob
import os

# === Static setup: Load reference data once ===

# Load genes2 from simulated file
genes2 = pd.read_csv(
    "/exports/eddie/scratch/s2556897/20250616_All-Tissues-NoDup_Random_Simulated_v2_Counts.txt",
    sep='\t', usecols=[0]
)["Geneid"]
set2 = set(genes2)

# Load reference gene map
ref_path = "/exports/eddie/scratch/s2556897/20250514_Published-cfRNA/20250405_Both_LessTissueV2_2Median-Unique.tsv"
reference = pd.read_csv(ref_path, sep='\t', index_col=False)
reference.columns.values[1] = "GeneName"
reference = reference.iloc[:, :2]
reference['GeneID_clean'] = reference['GeneID'].astype(str).str.split('.').str[0]
ref_map = dict(zip(reference['GeneName'], reference['GeneID_clean']))

# Load GTF annotation and extract gene name mapping
def extract_gtf_info(attr, key):
    match = re.search(f'{key} "([^"]+)"', attr)
    return match.group(1) if match else None

gtf = pd.read_csv("/exports/eddie/scratch/s2556897/gencode.v46.annotation.gtf", sep="\t", comment="#", header=None)
gtf.columns = ['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute']
genes_gtf = gtf[gtf['feature'] == 'gene'].copy()
genes_gtf['GeneID'] = genes_gtf['attribute'].apply(lambda x: extract_gtf_info(x, "gene_id"))
genes_gtf['GeneID_clean'] = genes_gtf['GeneID'].str.split('.').str[0]
genes_gtf['GeneName'] = genes_gtf['attribute'].apply(lambda x: extract_gtf_info(x, "gene_name"))
gtf_map = dict(zip(genes_gtf['GeneID_clean'], genes_gtf['GeneName']))

# === Process each BayesPrism matrix file ===

input_files = glob.glob("20250405_GeneID_LessTissueV2_*-Unique.tsv")

print(f"Found {len(input_files)} files to process.")

for filepath in input_files:
    print(f"Processing: {filepath}")

    # Step 1: Load genes from file
    genes1 = pd.read_csv(filepath, sep='\t', usecols=[0])["GeneID"]
    set1 = set(genes1)
    unique_to_file1 = set1 - set2

    # Step 4 & 5: Map via reference → GTF
    unmatched_to_geneid = {g: ref_map[g] for g in unique_to_file1 if g in ref_map}
    unmatched_to_gtf_name = {
        g: gtf_map[unmatched_to_geneid[g]] 
        for g in unmatched_to_geneid 
        if unmatched_to_geneid[g] in gtf_map
    }

    print(f"  Unmatched: {len(unique_to_file1)} → Remapped: {len(unmatched_to_gtf_name)}")

    # Step 6: Load full matrix and replace gene names
    bayes = pd.read_csv(filepath, sep='\t')
    bayes['GeneID'] = bayes['GeneID'].apply(lambda x: unmatched_to_gtf_name.get(x, x))

    # Step 7: Check for duplicate gene names and keep row with highest total count
    if bayes['GeneID'].duplicated().any():
        print("  Duplicate gene names found — deduplicating by row sum...")
        
        # Sum counts across all sample columns (excluding 'GeneID')
        bayes['row_sum'] = bayes.drop(columns='GeneID').sum(axis=1)

        # Keep the row with the highest sum for each duplicated GeneID
        bayes = bayes.sort_values('row_sum', ascending=False).drop_duplicates(subset='GeneID', keep='first')

        # Drop the helper column
        bayes = bayes.drop(columns='row_sum')

    # Save modified file
    out_path = filepath.replace("-Unique.tsv", "-withGTFNames.tsv")
    bayes.to_csv(out_path, sep='\t', index=False)
    print(f"  Saved to: {out_path}")


/tmp/ipykernel_820584/3671161587.py:17: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  reference = pd.read_csv(ref_path, sep='\t', index_col=False)


Found 3 files to process.
Processing: 20250405_GeneID_LessTissueV2_2Median-Unique.tsv
  Unmatched: 20100 → Remapped: 19358
  Duplicate gene names found — deduplicating by row sum...
  Saved to: 20250405_GeneID_LessTissueV2_2Median-withGTFNames.tsv
Processing: 20250405_GeneID_LessTissueV2_Sampling5-Unique.tsv
  Unmatched: 20100 → Remapped: 19358
  Duplicate gene names found — deduplicating by row sum...
  Saved to: 20250405_GeneID_LessTissueV2_Sampling5-withGTFNames.tsv
Processing: 20250405_GeneID_LessTissueV2_Sampling10-Unique.tsv
  Unmatched: 20100 → Remapped: 19358
  Duplicate gene names found — deduplicating by row sum...
  Saved to: 20250405_GeneID_LessTissueV2_Sampling10-withGTFNames.tsv


In [16]:
import pandas as pd
import re
import glob

# === Set paths ===
# File with all genes used in simulated data (used as reference for comparison)
simulated_counts_path = "/exports/eddie/scratch/s2556897/20250616_All-Tissues-NoDup_Random_Simulated_v2_Counts.txt"

# Reference file mapping GeneName to GeneID
ref_path = "/exports/eddie/scratch/s2556897/20250514_Published-cfRNA/20250405_Both_LessTissueV2_2Median-Unique.tsv"

# GTF file with gene annotations
gtf_path = "/exports/eddie/scratch/s2556897/gencode.v46.annotation.gtf"

# Directory with BayesPrism gene-renamed files
input_files = glob.glob("20250405_GeneID_LessTissueV2_*-withGTFNames.tsv")

# === Load simulated gene list (used as the "gold standard") ===
genes2 = pd.read_csv(simulated_counts_path, sep='\t', usecols=[0])["Geneid"]
set2 = set(genes2)

# === Load reference file with GeneName → GeneID_clean mapping ===
reference = pd.read_csv(ref_path, sep='\t', index_col=False)
reference.columns.values[1] = "GeneName"
reference = reference.iloc[:, :2]  # Keep only GeneID and GeneName
reference['GeneID_clean'] = reference['GeneID'].astype(str).str.split('.').str[0]
gene_name_to_id = dict(zip(reference['GeneName'], reference['GeneID_clean']))

# === Load and process the GTF file ===
gtf = pd.read_csv(gtf_path, sep="\t", comment="#", header=None, low_memory=False)
gtf.columns = ['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute']

# Function to extract version-less gene_id from attribute field
def extract_gene_id_clean(attr):
    match = re.search('gene_id "([^"]+)"', attr)
    if match:
        return match.group(1).split('.')[0]
    return None

# Add GeneID_clean column
gtf['GeneID_clean'] = gtf['attribute'].apply(extract_gene_id_clean)
genes_gtf = gtf[gtf['feature'] == 'gene']

# === Iterate over each renamed BayesPrism matrix and analyze gene overlaps ===
for file in input_files:
    print(f"\nProcessing file: {file}")
    
    # Load gene list from current file 
    try:
        genes1 = pd.read_csv(file, sep='\t', usecols=[0])["GeneID"]
    except Exception as e:
        print(f"  Skipped due to read error: {e}")
        continue

    set1 = set(genes1)

    # Identify shared and unique genes
    shared_genes = set1.intersection(set2)
    unique_to_file1 = set1 - set2

    print(f"  Total genes in file: {len(set1)}")
    print(f"  Shared with simulated: {len(shared_genes)}")
    print(f"  Not in simulated: {len(unique_to_file1)}")

    # Try to map unique genes to GeneID_clean using reference
    unmatched_genes_ids = [gene_name_to_id[gn] for gn in unique_to_file1 if gn in gene_name_to_id]

    # Filter GTF for those GeneID_clean values
    gtf_matched = genes_gtf[genes_gtf['GeneID_clean'].isin(unmatched_genes_ids)]
    print(f"  Unique genes mapped to GTF GeneIDs: {len(gtf_matched)}")


/tmp/ipykernel_820584/1644695843.py:23: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  reference = pd.read_csv(ref_path, sep='\t', index_col=False)



Processing file: 20250405_GeneID_LessTissueV2_Sampling10-withGTFNames.tsv
  Total genes in file: 54551
  Shared with simulated: 53809
  Not in simulated: 742
  Unique genes mapped to GTF GeneIDs: 0

Processing file: 20250405_GeneID_LessTissueV2_Sampling5-withGTFNames.tsv
  Total genes in file: 54551
  Shared with simulated: 53809
  Not in simulated: 742
  Unique genes mapped to GTF GeneIDs: 0

Processing file: 20250405_GeneID_LessTissueV2_2Median-withGTFNames.tsv
  Total genes in file: 54551
  Shared with simulated: 53809
  Not in simulated: 742
  Unique genes mapped to GTF GeneIDs: 0


In [17]:
!rm *-Unique.tsv
!rm CIBERSORTx-TOO-Matrix*/*.K999.txt